# Proyecto 5: Segmentación de Clientes mediante Clustering
## Notebook 5: Clustering Jerárquico

### Objetivo
Implementar Clustering Jerárquico (Hierarchical Clustering) como método alternativo para segmentación de clientes.

### Características del Clustering Jerárquico
**Definición:** Construye una jerarquía de clusters mediante procedimiento aglomerativo (bottom-up).

**Ventajas:**
- No requiere especificar k previamente
- Produce dendrograma informativo
- Flexible en la selección de número de clusters
- Múltiples métodos de vinculación

**Métodos de Vinculación:**
1. **Single**: Mínima distancia entre puntos
2. **Complete**: Máxima distancia entre puntos
3. **Average**: Distancia promedio
4. **Ward**: Minimiza varianza intra-cluster

## 1. Importación y Preparación

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Librerías importadas")

In [ ]:
# Cargar y preparar datos
ruta_datos = '../data/Wholesale customers data.csv'
df_original = pd.read_csv(ruta_datos)

variables_numericas = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
scaler = StandardScaler()
df_estandarizado = df_original.copy()
df_estandarizado[variables_numericas] = scaler.fit_transform(df_original[variables_numericas])

X = df_estandarizado[variables_numericas].values

print(f"✓ Datos cargados: {X.shape}")

## 2. Comparación de Métodos de Vinculación

In [ ]:
# Calcular matrices de vinculación con diferentes métodos
métodos_vinculación = ['single', 'complete', 'average', 'ward']
matrices_vinculacion = {}

print("Calculando matrices de vinculación...\n")

for metodo in métodos_vinculación:
    print(f"  • {metodo.upper()}...", end="")
    Z = linkage(X, method=metodo)
    matrices_vinculacion[metodo] = Z
    print(" ✓")

print("\n✓ Todas las matrices calculadas")

## 3. Dendrogramas - Visualización Completa

In [ ]:
# Visualizar dendrogramas para cada método
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, metodo in enumerate(métodos_vinculación):
    dendrogram(matrices_vinculacion[metodo], ax=axes[idx], no_labels=True, color_threshold=0)
    axes[idx].set_title(f'Dendrogram - Método: {metodo.upper()}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Índice de Muestra')
    axes[idx].set_ylabel('Distancia')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Dendrogramas generados para todos los métodos")

## 4. Dendrogram Detallado con Ward (Método Recomendado)

In [ ]:
# Usar método Ward (mejor para clustering)
Z_ward = matrices_vinculacion['ward']

# Crear dendrogram con color threshold
plt.figure(figsize=(16, 8))
dendrogram(Z_ward, 
          no_labels=True,
          color_threshold=30,  # Línea de corte para identificar clusters
          above_threshold_color='gray')
plt.axhline(y=30, color='red', linestyle='--', linewidth=2, label='Corte para k=3')
plt.xlabel('Índice de Muestra', fontweight='bold', fontsize=11)
plt.ylabel('Distancia', fontweight='bold', fontsize=11)
plt.title('Dendrogram - Clustering Jerárquico con Método Ward\n(Distancia de corte para k=3 marcada en rojo)', 
         fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("✓ Dendrogram Ward detallado generado")

## 5. Análisis de Distancias de Corte

In [ ]:
# Analizar las últimas distancias de fusión
ultimas_fusiones = Z_ward[-15:, 2]  # Últimas 15 distancias
diferencias = np.diff(ultimas_fusiones)

print("ANÁLISIS DE DISTANCIAS DE FUSIÓN (Últimas 15)\n" + "="*70)
print(f"\n{'Fusión Número':<20} {'Distancia':<20} {'Diferencia':<20}")
print("-" * 70)

for i, (dist, diff) in enumerate(zip(ultimas_fusiones, list(diferencias) + [np.nan])):
    num_fusion = len(Z_ward) - 15 + i + 1
    if np.isnan(diff):
        print(f"{num_fusion:<20} {dist:<20.2f} {'N/A':<20}")
    else:
        print(f"{num_fusion:<20} {dist:<20.2f} {diff:<20.2f}")

# Identificar salto más grande
mayor_salto_idx = np.argmax(diferencias)
salto_mayor = diferencias[mayor_salto_idx]
print(f"\nMayor salto de distancia: {salto_mayor:.2f} (entre fusión {len(Z_ward)-15+mayor_salto_idx} y {len(Z_ward)-15+mayor_salto_idx+1})")
print(f"Esto sugiere k ≈ {len(Z_ward) - (len(Z_ward) - 15 + mayor_salto_idx)} clusters")

## 6. Obtener Clusters con k=3

In [ ]:
# Obtener clusters usando fcluster
k_optimo = 3
clusters_jerarquico = fcluster(Z_ward, k_optimo, criterion='maxclust')

# Agregar al dataframe
df_estandarizado['Jerarquico_3'] = clusters_jerarquico - 1  # Convertir a 0-indexed
df_original['Jerarquico_3'] = clusters_jerarquico - 1

print(f"✓ Clusters jerárquicos obtenidos con k={k_optimo}")
print(f"\nDistribución de clusters:")
print(df_estandarizado['Jerarquico_3'].value_counts().sort_index())

# Calcular métricas
silhouette_jer = silhouette_score(X, clusters_jerarquico - 1)
davies_bouldin_jer = davies_bouldin_score(X, clusters_jerarquico - 1)
calinski_jer = calinski_harabasz_score(X, clusters_jerarquico - 1)

print(f"\nMétricas de Calidad:")
print(f"  • Silhouette Score: {silhouette_jer:.3f}")
print(f"  • Davies-Bouldin Index: {davies_bouldin_jer:.3f}")
print(f"  • Calinski-Harabasz: {calinski_jer:.2f}")

## 7. Visualización de Clusters en PCA

In [ ]:
# Aplicar PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Visualizar clusters
plt.figure(figsize=(12, 8))

colores_cluster = ['#FF6B6B', '#4ECDC4', '#45B7D1']
labels_cluster = clusters_jerarquico - 1

for i in range(k_optimo):
    mask = labels_cluster == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], 
               c=colores_cluster[i], label=f'Cluster {i}',
               s=60, alpha=0.7, edgecolors='black', linewidth=0.5)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontweight='bold', fontsize=11)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontweight='bold', fontsize=11)
plt.title('Clustering Jerárquico (Método Ward, k=3)\nVisualización en 2D usando PCA', 
         fontsize=13, fontweight='bold', pad=15)
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Visualización en PCA completada")

## 8. Análisis de Características por Cluster

In [ ]:
print("ANÁLISIS DE CARACTERÍSTICAS POR CLUSTER (JERÁRQUICO)\n" + "="*80)

for cluster in range(k_optimo):
    df_cluster = df_original[df_original['Jerarquico_3'] == cluster]
    
    print(f"\nCluster {cluster} (n={len(df_cluster)} | {len(df_cluster)/len(df_original)*100:.1f}%):")
    print("-" * 80)
    
    # Gasto promedio
    print("Gasto promedio:")
    for var in variables_numericas:
        print(f"  • {var:20s}: ${df_cluster[var].mean():>10,.0f}")
    
    # Composición
    print(f"\nCanal predominante: {df_cluster['Channel'].mode()[0]} ({df_cluster['Channel'].value_counts().iloc[0]} clientes)")
    print(f"Región predominante: {df_cluster['Region'].mode()[0]}")

## 9. Comparación: Clustering Jerárquico vs K-means

In [ ]:
# Cargar clusters de K-means
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X)

# Crear matriz de confusión
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(labels_cluster, labels_kmeans)
nmi = normalized_mutual_info_score(labels_cluster, labels_kmeans)

print("\nCOMPARACIÓN CLUSTERING JERÁRQUICO vs K-MEANS\n" + "="*70)
print(f"\nAdjusted Rand Index (ARI): {ari:.3f}")
print(f"  • Rango: -1 a 1")
print(f"  • 1 = particiones idénticas")
print(f"  • 0 = asignaciones aleatorias")

print(f"\nNormalized Mutual Information (NMI): {nmi:.3f}")
print(f"  • Rango: 0 a 1")
print(f"  • 1 = partidiones idénticas")
print(f"  • 0 = particiones independientes")

if abs(ari) > 0.7:
    print(f"\n✓ Los dos métodos producen particiones muy similares")
else:
    print(f"\n⚠ Los dos métodos producen particiones algo diferentes")

In [ ]:
# Tabla de comparación de métricas
comparacion = pd.DataFrame({
    'Métrica': ['Silhouette Score', 'Davies-Bouldin', 'Calinski-Harabasz'],
    'K-means': [0.4179, 1.4584, 231.74],  # Valores del notebook anterior
    'Jerárquico': [silhouette_jer, davies_bouldin_jer, calinski_jer]
})

print("\nTABLA COMPARATIVA DE MÉTRICAS:")
print(comparacion.to_string(index=False))

# Visualizar comparación
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metodos = ['K-means', 'Jerárquico']
valores_silhouette = [0.4179, silhouette_jer]
valores_db = [1.4584, davies_bouldin_jer]
valores_ch = [231.74, calinski_jer]

# Silhouette (mayor es mejor)
axes[0].bar(metodos, valores_silhouette, color=['#FF6B6B', '#4ECDC4'], alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Score', fontweight='bold')
axes[0].set_title('Silhouette Score (↑ mejor)', fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(valores_silhouette):
    axes[0].text(i, v + 0.01, f"{v:.3f}", ha='center', va='bottom', fontweight='bold')

# Davies-Bouldin (menor es mejor)
axes[1].bar(metodos, valores_db, color=['#FF6B6B', '#4ECDC4'], alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Index', fontweight='bold')
axes[1].set_title('Davies-Bouldin (↓ mejor)', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(valores_db):
    axes[1].text(i, v + 0.02, f"{v:.3f}", ha='center', va='bottom', fontweight='bold')

# Calinski-Harabasz (mayor es mejor)
axes[2].bar(metodos, valores_ch, color=['#FF6B6B', '#4ECDC4'], alpha=0.7, edgecolor='black')
axes[2].set_ylabel('Score', fontweight='bold')
axes[2].set_title('Calinski-Harabasz (↑ mejor)', fontweight='bold')
axes[2].grid(axis='y', alpha=0.3)
for i, v in enumerate(valores_ch):
    axes[2].text(i, v + 5, f"{v:.1f}", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Matriz de Distancias

In [ ]:
# Calcular matriz de distancias condensadas
dist_matrix = pdist(X, metric='euclidean')
dist_pairwise = squareform(dist_matrix)

# Visualizar una submatriz
submatrix_size = 20
plt.figure(figsize=(10, 9))
sns.heatmap(dist_pairwise[:submatrix_size, :submatrix_size], 
            cmap='viridis', cbar_kws={'label': 'Distancia Euclidiana'},
            square=True, linewidths=0.5)
plt.title(f'Matriz de Distancias (primeras {submatrix_size} muestras)', fontsize=12, fontweight='bold')
plt.xlabel('Índice de Muestra')
plt.ylabel('Índice de Muestra')
plt.tight_layout()
plt.show()

## 11. Dendrograma Truncado para Mayor Claridad

In [ ]:
# Dendrogram truncado (últimas 30 merges)
plt.figure(figsize=(14, 8))
dendrogram(Z_ward,
          truncate_mode='lastp',
          p=30,
          leaf_label_func=lambda id: f"Cluster {labels_cluster[id]}" if id < len(labels_cluster) else f"Merge {id}",
          leaf_rotation=90,
          leaf_font_size=10,
          show_contracted=True,
          color_threshold=30)
plt.axhline(y=30, color='red', linestyle='--', linewidth=2, label='Corte para k=3')
plt.xlabel('Índice de Muestra / Clusters', fontweight='bold')
plt.ylabel('Distancia', fontweight='bold')
plt.title('Dendrogram Truncado - Últimas 30 Fusiones\n(con indicador de clusters finales)', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 12. Evaluación con Diferentes Números de Clusters

In [ ]:
# Evaluar diferentes k's
k_values = range(2, 9)
resultados_jer = {}

for k in k_values:
    labels_k = fcluster(Z_ward, k, criterion='maxclust') - 1
    
    silhouette = silhouette_score(X, labels_k)
    davies_bouldin = davies_bouldin_score(X, labels_k)
    calinski = calinski_harabasz_score(X, labels_k)
    
    resultados_jer[k] = {
        'silhouette': silhouette,
        'davies_bouldin': davies_bouldin,
        'calinski': calinski
    }

print("EVALUACIÓN POR NÚMERO DE CLUSTERS (JERÁRQUICO)\n" + "="*80)
print(f"\n{'k':<5} {'Silhouette':<15} {'Davies-B.':<15} {'Calinski':<15}")
print("-" * 80)

for k in sorted(resultados_jer.keys()):
    res = resultados_jer[k]
    print(f"{k:<5} {res['silhouette']:<15.3f} {res['davies_bouldin']:<15.3f} {res['calinski']:<15.2f}")

## 13. Resumen y Conclusiones

In [ ]:
resumen = f"""
{'='*80}
RESUMEN - CLUSTERING JERÁRQUICO
{'='*80}

1. METODOLOGÍA:
   ✓ Algoritmo: Clustering Jerárquico Aglomerativo
   ✓ Método de Vinculación: Ward (minimiza varianza intra-cluster)
   ✓ Distancia: Euclidiana
   ✓ Número de clusters seleccionado: k=3

2. RESULTADOS PRINCIPALES:
   • Silhouette Score: {silhouette_jer:.3f}
   • Davies-Bouldin Index: {davies_bouldin_jer:.3f}
   • Calinski-Harabasz Score: {calinski_jer:.2f}

3. DISTRIBUCIÓN DE CLUSTERS:
   • Cluster 0: {len(df_original[df_original['Jerarquico_3']==0])} clientes
   • Cluster 1: {len(df_original[df_original['Jerarquico_3']==1])} clientes
   • Cluster 2: {len(df_original[df_original['Jerarquico_3']==2])} clientes

4. COMPARACIÓN CON K-MEANS:
   • Adjusted Rand Index (ARI): {ari:.3f}
   • Normalized Mutual Information (NMI): {nmi:.3f}
   • Conclusión: Particiones {'muy similares' if abs(ari) > 0.7 else 'moderadamente similares'}

5. VENTAJAS DEL MÉTODO:
   ✓ Dendrogram informativo
   ✓ No requiere especificar k previamente
   ✓ Flexible en selección de número de clusters
   ✓ Resultados jerárquicos e interpretables

6. DESVENTAJAS:
   ⚠ Más lento computacionalmente que K-means
   ⚠ Sensible al método de vinculación
   ⚠ No lleva a mejores métricas que K-means

7. RECOMENDACIÓN:
   → Ambos métodos (K-means y Jerárquico) producen resultados similares
   → Considerar DBSCAN para evaluación comparativa completa
   → Decisión final basada en interpretabilidad de negocio

{'='*80}
"""

print(resumen)

In [ ]:
print("✓ Análisis de Clustering Jerárquico completado")
print("✓ Resultados listos para comparación con DBSCAN")